# Kaggle vLLM Qwen Notebook (Hardened, Local-First)

This notebook is optimized for Kaggle single-GPU stability (T4/L4 style).
It avoids unnecessary reinstalls, reuses local model artifacts, logs memory, and provides restart-safe cleanup.

Run cells in order:
1. Setup config and runtime state
2. Helper functions (install/model/memory/cleanup)
3. Install dependencies (guarded + pinned)
4. Model lock + auth + prepare `/kaggle/working/model`
5. Initialize local vLLM engine (Kaggle-safe profile)
6. Interactive test calls with `generate(...)`
7. Runtime diagnostics snapshot
8. Cleanup/reset before re-running heavy cells

Notes:
- Default path is local in-notebook inference for maximum reliability.
- External tunnel/API exposure is intentionally not required for interactive testing.
- Set environment variables before running to override defaults.

In [ ]:
import gc
import importlib.metadata as importlib_metadata
import json
import os
import re
import shutil
import subprocess
import sys
import time
import traceback
import urllib.request
from pathlib import Path
from typing import Any, Dict, Optional

MODEL_ID = "Deign86/deped-math-qwen2.5-7b-deped-math-merged"
DISALLOWED_MODEL_IDS = {
    "Deign86/deped-math-qwen2.5-7b-checkpoint-700-lora",
    "Qwen/Qwen2.5-7B-Instruct",
}

HOST = "0.0.0.0"
PORT = int(os.getenv("API_PORT", "8000"))

# Kaggle-safe defaults: prioritize stability and avoid aggressive VRAM overcommit.
DTYPE = os.getenv("VLLM_DTYPE", "float16").strip() or "float16"
MAX_MODEL_LEN = int(os.getenv("VLLM_MAX_MODEL_LEN", "2048"))
GPU_MEMORY_UTILIZATION = float(os.getenv("VLLM_GPU_MEMORY_UTILIZATION", "0.75"))
TENSOR_PARALLEL_SIZE = int(os.getenv("VLLM_TENSOR_PARALLEL_SIZE", "1"))
MAX_NUM_SEQS = int(os.getenv("VLLM_MAX_NUM_SEQS", "2"))
MAX_NUM_BATCHED_TOKENS = int(os.getenv("VLLM_MAX_NUM_BATCHED_TOKENS", "4096"))
CPU_OFFLOAD_GB = float(os.getenv("VLLM_CPU_OFFLOAD_GB", "8"))
# Use API-level backend configuration for vLLM 0.19+.
ATTENTION_BACKEND = os.getenv("VLLM_ATTENTION_BACKEND", "TRITON_ATTN").strip() or "TRITON_ATTN"
DISABLE_FLASHINFER = os.getenv("VLLM_DISABLE_FLASHINFER", "1").strip().lower() in {"1", "true", "yes"}
ENFORCE_EAGER = os.getenv("VLLM_ENFORCE_EAGER", "1").strip().lower() in {"1", "true", "yes"}
TRUST_REMOTE_CODE = os.getenv("VLLM_TRUST_REMOTE_CODE", "0").strip().lower() in {"1", "true", "yes"}

# Optional quantization. Leave empty for full-precision/float16 checkpoints.
QUANTIZATION = os.getenv("VLLM_QUANTIZATION", "").strip() or None

MAX_GENERATE_TOKENS = int(os.getenv("MAX_GENERATE_TOKENS", "384"))

# Keep these defined even in local-first mode because legacy helper defaults reference them.
CLOUDFLARED_MAX_ATTEMPTS = int(os.getenv("CLOUDFLARED_MAX_ATTEMPTS", "3"))
CLOUDFLARED_ATTEMPT_TIMEOUT = int(os.getenv("CLOUDFLARED_ATTEMPT_TIMEOUT", "60"))

WORK_DIR = Path("/kaggle/working")
MODEL_LOCAL_DIR = WORK_DIR / "model"
HF_CACHE_DIR = WORK_DIR / "hf-cache"
RUNTIME_STATUS_PATH = WORK_DIR / "api_runtime_status.json"
CLOUDFLARED_LOG_PATH = WORK_DIR / "cloudflared.log"

# Optional legacy process handles (used only if you later add API-server/tunnel cells).
vllm_proc = None
cloudflared_proc = None
public_url = ""
openai_base_url = ""

RUNTIME: Dict[str, Any] = {
    "dependencies_ready": False,
    "model_prepared": False,
    "engine_ready": False,
    "engine": None,
    "model_path": None,
    "last_init_seconds": None,
    "last_profile": {},
}

print(
    json.dumps(
        {
            "MODEL_ID": MODEL_ID,
            "MODEL_LOCAL_DIR": str(MODEL_LOCAL_DIR),
            "DTYPE": DTYPE,
            "MAX_MODEL_LEN": MAX_MODEL_LEN,
            "GPU_MEMORY_UTILIZATION": GPU_MEMORY_UTILIZATION,
            "MAX_NUM_SEQS": MAX_NUM_SEQS,
            "CPU_OFFLOAD_GB": CPU_OFFLOAD_GB,
            "ATTENTION_BACKEND": ATTENTION_BACKEND,
            "DISABLE_FLASHINFER": DISABLE_FLASHINFER,
            "ENFORCE_EAGER": ENFORCE_EAGER,
            "QUANTIZATION": QUANTIZATION,
        },
        indent=2,
    )
)

In [ ]:
from packaging.specifiers import SpecifierSet
from packaging.version import InvalidVersion, Version


def stage(title: str) -> None:
    print(f"\n{'=' * 12} {title} {'=' * 12}")


def run_cli(
    command: list[str],
    check: bool = True,
    env: Optional[Dict[str, str]] = None,
    quiet: bool = False,
) -> subprocess.CompletedProcess[str]:
    cmd_env = os.environ.copy()
    if env:
        cmd_env.update(env)

    if not quiet:
        print("[cli]", " ".join(command))

    proc = subprocess.run(
        command,
        check=False,
        text=True,
        capture_output=True,
        env=cmd_env,
    )

    if proc.stdout.strip() and not quiet:
        print(proc.stdout.strip())
    if proc.stderr.strip() and not quiet:
        print(proc.stderr.strip())

    if check and proc.returncode != 0:
        raise RuntimeError(f"Command failed ({proc.returncode}): {' '.join(command)}")

    return proc


def get_installed_version(dist_name: str) -> Optional[str]:
    try:
        return importlib_metadata.version(dist_name)
    except importlib_metadata.PackageNotFoundError:
        return None


def version_satisfies(installed: Optional[str], spec: str) -> bool:
    if installed is None:
        return False
    try:
        return Version(installed) in SpecifierSet(spec)
    except (InvalidVersion, ValueError):
        return False


def pip_install(packages: list[str], no_deps: bool = False) -> None:
    if not packages:
        return

    cmd = [sys.executable, "-m", "pip", "install", "-q", "--no-cache-dir"]
    if no_deps:
        # Safe for narrow compatibility pins where we only need to constrain versions.
        cmd.append("--no-deps")
    cmd.extend(packages)
    run_cli(cmd, check=True)


def pip_uninstall(packages: list[str]) -> None:
    if not packages:
        return

    cmd = [sys.executable, "-m", "pip", "uninstall", "-y"]
    cmd.extend(packages)
    run_cli(cmd, check=False)


def enforce_protobuf_runtime_compat() -> Dict[str, Any]:
    status: Dict[str, Any] = {
        "protobuf": get_installed_version("protobuf"),
        "patched": False,
    }
    try:
        from google.protobuf import message_factory

        factory_cls = getattr(message_factory, "MessageFactory", None)
        if factory_cls is None:
            status["warning"] = "MessageFactory unavailable"
            return status

        if hasattr(factory_cls, "GetPrototype"):
            return status

        message_class_loader = getattr(message_factory, "GetMessageClass", None)
        if callable(message_class_loader):

            def _get_prototype(self, descriptor):
                return message_class_loader(descriptor)

            setattr(factory_cls, "GetPrototype", _get_prototype)
            status["patched"] = True
            status["mode"] = "GetMessageClass"
            return status

        from google.protobuf import symbol_database

        def _get_prototype_fallback(self, descriptor):
            return symbol_database.Default().GetPrototype(descriptor)

        setattr(factory_cls, "GetPrototype", _get_prototype_fallback)
        status["patched"] = True
        status["mode"] = "symbol_database"
    except Exception as exc:
        status["warning"] = str(exc)

    return status


def assert_model_lock() -> None:
    env_model_id = os.getenv("MODEL_ID", "").strip()
    if env_model_id and env_model_id != MODEL_ID:
        raise RuntimeError(
            f"MODEL_ID override is not allowed. Expected {MODEL_ID}, got {env_model_id}."
        )
    if MODEL_ID in DISALLOWED_MODEL_IDS:
        raise RuntimeError(f"Configured MODEL_ID is disallowed: {MODEL_ID}")


def configure_hf_auth() -> Dict[str, str]:
    token = os.getenv("HF_TOKEN", "").strip()
    if not token:
        return {"used_token": "false", "source": "missing"}

    os.environ["HF_TOKEN"] = token
    os.environ["HUGGING_FACE_HUB_TOKEN"] = token

    try:
        from huggingface_hub import login

        login(token=token, add_to_git_credential=False)
        return {"used_token": "true", "source": "HF_TOKEN", "method": "huggingface_hub.login"}
    except Exception as exc:
        print(f"HF login warning: {exc}")
        return {
            "used_token": "true",
            "source": "HF_TOKEN",
            "method": "env_only",
            "warning": str(exc),
        }


def _extract_special_token(item: object) -> str:
    if isinstance(item, str):
        return item.strip()
    if isinstance(item, dict):
        for key in ["token", "content", "value", "text"]:
            value = item.get(key)
            if isinstance(value, str) and value.strip():
                return value.strip()
    return ""


def is_model_artifact_complete(model_dir: Path) -> bool:
    if not model_dir.exists():
        return False

    has_config = (model_dir / "config.json").exists()
    has_tokenizer = (model_dir / "tokenizer_config.json").exists() or (model_dir / "tokenizer.json").exists()
    has_weights = bool(list(model_dir.glob("*.safetensors")) or list(model_dir.glob("*.bin")))
    return has_config and has_tokenizer and has_weights


def prepare_model_artifacts(force_download: bool = False) -> Path:
    from huggingface_hub import snapshot_download

    MODEL_LOCAL_DIR.mkdir(parents=True, exist_ok=True)
    HF_CACHE_DIR.mkdir(parents=True, exist_ok=True)

    if is_model_artifact_complete(MODEL_LOCAL_DIR) and not force_download:
        print(f"Model already present, skipping download: {MODEL_LOCAL_DIR}")
    else:
        print("Downloading or refreshing model artifacts...")
        snapshot_download(
            repo_id=MODEL_ID,
            local_dir=str(MODEL_LOCAL_DIR),
            cache_dir=str(HF_CACHE_DIR),
            resume_download=True,
        )

    tokenizer_config_path = MODEL_LOCAL_DIR / "tokenizer_config.json"
    if tokenizer_config_path.exists():
        try:
            tokenizer_config = json.loads(tokenizer_config_path.read_text(encoding="utf-8"))
            changed = False
            extra_special_tokens = tokenizer_config.get("extra_special_tokens")

            if isinstance(extra_special_tokens, list):
                extracted_tokens = [
                    token
                    for token in (_extract_special_token(item) for item in extra_special_tokens)
                    if token
                ]
                additional_tokens = tokenizer_config.get("additional_special_tokens", [])
                if not isinstance(additional_tokens, list):
                    additional_tokens = []
                merged = list(dict.fromkeys([*additional_tokens, *extracted_tokens]))
                tokenizer_config["additional_special_tokens"] = merged
                tokenizer_config["extra_special_tokens"] = {}
                changed = True
            elif extra_special_tokens is not None and not isinstance(extra_special_tokens, dict):
                tokenizer_config["extra_special_tokens"] = {}
                changed = True

            if changed:
                tokenizer_config_path.write_text(json.dumps(tokenizer_config, indent=2), encoding="utf-8")
        except Exception as exc:
            print(f"Tokenizer config normalization skipped: {exc}")

    if not is_model_artifact_complete(MODEL_LOCAL_DIR):
        raise RuntimeError("Model artifacts are incomplete after prepare step.")

    RUNTIME["model_prepared"] = True
    RUNTIME["model_path"] = MODEL_LOCAL_DIR
    return MODEL_LOCAL_DIR


def get_cpu_memory_snapshot() -> Dict[str, Any]:
    snapshot: Dict[str, Any] = {}
    try:
        import psutil

        vm = psutil.virtual_memory()
        snapshot = {
            "total_gb": round(vm.total / (1024**3), 2),
            "available_gb": round(vm.available / (1024**3), 2),
            "used_gb": round(vm.used / (1024**3), 2),
            "percent": vm.percent,
        }
    except Exception as exc:
        snapshot = {"warning": f"psutil unavailable: {exc}"}
    return snapshot


def get_gpu_memory_snapshot() -> Dict[str, Any]:
    snapshot: Dict[str, Any] = {"cuda_available": False}
    try:
        import torch

        if not torch.cuda.is_available():
            return snapshot

        device_index = torch.cuda.current_device()
        total_bytes = torch.cuda.get_device_properties(device_index).total_memory
        free_bytes, total_check = torch.cuda.mem_get_info()
        allocated = torch.cuda.memory_allocated(device_index)
        reserved = torch.cuda.memory_reserved(device_index)
        snapshot = {
            "cuda_available": True,
            "device": torch.cuda.get_device_name(device_index),
            "allocated_gb": round(allocated / (1024**3), 2),
            "reserved_gb": round(reserved / (1024**3), 2),
            "free_gb": round(free_bytes / (1024**3), 2),
            "total_gb": round(total_bytes / (1024**3), 2),
            "driver_total_gb": round(total_check / (1024**3), 2),
            "peak_allocated_gb": round(torch.cuda.max_memory_allocated(device_index) / (1024**3), 2),
        }
    except Exception as exc:
        snapshot = {"cuda_available": False, "warning": str(exc)}
    return snapshot


def log_memory(tag: str) -> Dict[str, Any]:
    payload = {
        "tag": tag,
        "timestamp": time.strftime("%Y-%m-%d %H:%M:%S"),
        "cpu": get_cpu_memory_snapshot(),
        "gpu": get_gpu_memory_snapshot(),
    }
    print(json.dumps(payload, indent=2))
    return payload


def write_runtime_status(payload: Dict[str, Any]) -> None:
    RUNTIME_STATUS_PATH.write_text(json.dumps(payload, indent=2), encoding="utf-8")


def terminate_process(proc: Optional[subprocess.Popen[Any]], name: str) -> None:
    if proc is None or proc.poll() is not None:
        return
    print(f"Stopping {name}...")
    proc.terminate()
    try:
        proc.wait(timeout=20)
    except Exception:
        proc.kill()


def cleanup_engine(reset_cuda: bool = True, stop_legacy_processes: bool = True) -> None:
    global cloudflared_proc, openai_base_url, public_url, vllm_proc

    if stop_legacy_processes:
        terminate_process(cloudflared_proc, "cloudflared")
        terminate_process(vllm_proc, "vLLM api server")
        cloudflared_proc = None
        vllm_proc = None

    engine = RUNTIME.get("engine")
    if engine is not None:
        shutdown_fn = getattr(engine, "shutdown", None)
        if callable(shutdown_fn):
            try:
                shutdown_fn()
            except Exception as exc:
                print(f"Engine shutdown warning: {exc}")

    RUNTIME["engine"] = None
    RUNTIME["engine_ready"] = False
    RUNTIME["last_init_seconds"] = None

    gc.collect()

    if reset_cuda:
        try:
            import torch

            if torch.cuda.is_available():
                torch.cuda.empty_cache()
                if hasattr(torch.cuda, "ipc_collect"):
                    torch.cuda.ipc_collect()
        except Exception as exc:
            print(f"CUDA cleanup warning: {exc}")

    public_url = ""
    openai_base_url = ""
    log_memory("after_cleanup")
    print("Cleanup complete.")


DEPENDENCY_PLAN = [
    {
        "dist": "vllm",
        "spec": ">=0.19.0,<0.20.0",
        "pip": "vllm==0.19.0",
        "group": "core",
    },
    {
        "dist": "huggingface-hub",
        "spec": ">=0.23.0,<1.0.0",
        "pip": "huggingface_hub>=0.23.0,<1.0.0",
        "group": "core",
    },
    {
        "dist": "requests",
        "spec": ">=2.31.0,<3.0.0",
        "pip": "requests>=2.31.0,<3.0.0",
        "group": "core",
    },
    {
        "dist": "packaging",
        "spec": ">=24.0",
        "pip": "packaging>=24.0",
        "group": "core",
    },
    {
        "dist": "protobuf",
        "spec": ">=4.25.3,<6.0.0",
        "pip": "protobuf==5.29.5",
        "group": "compat",
    },
    {
        "dist": "opentelemetry-api",
        "spec": ">=1.36.0,<1.40.0",
        "pip": "opentelemetry-api==1.39.0",
        "group": "compat",
    },
    {
        "dist": "opentelemetry-sdk",
        "spec": ">=1.36.0,<1.40.0",
        "pip": "opentelemetry-sdk==1.39.0",
        "group": "compat",
    },
    {
        "dist": "torch",
        "spec": ">=2.4.0,<2.7.0",
        "pip": "torch>=2.4.0,<2.7.0",
        "group": "heavy",
    },
]


def install_dependencies(force: bool = False) -> None:
    stage("Install Dependencies")

    install_groups = {"core": [], "compat": [], "heavy": []}
    report: Dict[str, Dict[str, Any]] = {}

    for item in DEPENDENCY_PLAN:
        dist = item["dist"]
        spec = item["spec"]
        installed = get_installed_version(dist)
        is_ok = version_satisfies(installed, spec)
        needs_install = force or not is_ok

        report[dist] = {
            "installed": installed,
            "required": spec,
            "status": "ok" if is_ok and not force else "needs_action",
        }

        if needs_install and item["group"] != "compat":
            install_groups[item["group"]].append(item["pip"])

    if install_groups["core"]:
        print("Installing core packages:", install_groups["core"])
        pip_install(install_groups["core"], no_deps=False)
    else:
        print("Core packages already satisfy pinned ranges.")

    compat_packages = []
    for item in DEPENDENCY_PLAN:
        if item["group"] != "compat":
            continue
        installed = get_installed_version(item["dist"])
        if force or not version_satisfies(installed, item["spec"]):
            compat_packages.append(item["pip"])

    if compat_packages:
        print("Applying compatibility pins with --no-deps:", compat_packages)
        run_cli(
            [
                sys.executable,
                "-m",
                "pip",
                "install",
                "-q",
                "--no-cache-dir",
                "--upgrade",
                "--force-reinstall",
                "--no-deps",
                *compat_packages,
            ],
            check=True,
        )
    else:
        print("Compatibility pins already satisfied.")

    if install_groups["heavy"]:
        torch_installed = get_installed_version("torch")
        allow_heavy = os.getenv("ALLOW_HEAVY_REINSTALL", "0").strip().lower() in {"1", "true", "yes"}
        if torch_installed is None or allow_heavy:
            print("Installing torch pin:", install_groups["heavy"])
            pip_install(install_groups["heavy"], no_deps=False)
        else:
            print(
                "Torch version is outside preferred range but heavy reinstall is disabled. "
                "Set ALLOW_HEAVY_REINSTALL=1 to force a torch reinstall.",
            )

    if DISABLE_FLASHINFER:
        print("Disabling FlashInfer wheels for Kaggle linker compatibility.")
        pip_uninstall(["flashinfer", "flashinfer-python"])

    verification: Dict[str, str] = {}
    for item in DEPENDENCY_PLAN:
        dist = item["dist"]
        spec = item["spec"]
        installed = get_installed_version(dist)
        ok = version_satisfies(installed, spec)
        verification[dist] = f"{installed} ({'ok' if ok else 'check'})"

    compat_failures = [
        item["dist"]
        for item in DEPENDENCY_PLAN
        if item["group"] == "compat"
        and not version_satisfies(get_installed_version(item["dist"]), item["spec"])
    ]
    if compat_failures:
        raise RuntimeError(
            "Compatibility pins failed to apply for: " + ", ".join(compat_failures)
        )

    RUNTIME["dependencies_ready"] = True
    write_runtime_status({
        "phase": "dependencies",
        "verification": verification,
    })
    print(json.dumps(verification, indent=2))


def build_runtime_profile() -> Dict[str, Any]:
    profile = {
        "dtype": DTYPE,
        "max_model_len": MAX_MODEL_LEN,
        "gpu_memory_utilization": GPU_MEMORY_UTILIZATION,
        "tensor_parallel_size": TENSOR_PARALLEL_SIZE,
        "max_num_seqs": MAX_NUM_SEQS,
        "max_num_batched_tokens": MAX_NUM_BATCHED_TOKENS,
        "cpu_offload_gb": CPU_OFFLOAD_GB,
        "enforce_eager": ENFORCE_EAGER,
        "trust_remote_code": TRUST_REMOTE_CODE,
        "quantization": QUANTIZATION,
        "attention_backend": ATTENTION_BACKEND,
    }

    try:
        import torch

        if torch.cuda.is_available():
            gpu_name = torch.cuda.get_device_name(0).lower()
            if "t4" in gpu_name:
                profile["gpu_memory_utilization"] = min(profile["gpu_memory_utilization"], 0.75)
                profile["max_model_len"] = min(profile["max_model_len"], 2048)
                profile["max_num_seqs"] = min(profile["max_num_seqs"], 2)
            elif "l4" in gpu_name:
                profile["gpu_memory_utilization"] = min(profile["gpu_memory_utilization"], 0.82)
                profile["max_num_seqs"] = min(profile["max_num_seqs"], 4)
    except Exception as exc:
        print(f"GPU profile detection warning: {exc}")

    return profile


def create_vllm_engine(force_reinit: bool = False, run_warmup: bool = True):
    if RUNTIME.get("engine_ready") and not force_reinit:
        print("Engine already initialized. Reusing existing instance.")
        return RUNTIME["engine"]

    if force_reinit:
        cleanup_engine(reset_cuda=True, stop_legacy_processes=True)

    if not RUNTIME.get("dependencies_ready"):
        install_dependencies(force=False)

    if not RUNTIME.get("model_prepared") or RUNTIME.get("model_path") is None:
        prepare_model_artifacts(force_download=False)

    model_path = Path(RUNTIME["model_path"])
    profile = build_runtime_profile()
    RUNTIME["last_profile"] = profile

    os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
    os.environ["VLLM_ATTENTION_BACKEND"] = profile["attention_backend"]

    if DISABLE_FLASHINFER:
        os.environ["VLLM_USE_FLASHINFER_SAMPLER"] = "0"
        os.environ["VLLM_BLOCKSCALE_FP8_GEMM_FLASHINFER"] = "0"
        os.environ["VLLM_ALLREDUCE_USE_FLASHINFER"] = "0"
        os.environ["VLLM_USE_FLASHINFER_MOE_FP16"] = "0"
        os.environ["VLLM_USE_FLASHINFER_MOE_FP8"] = "0"
        os.environ["VLLM_USE_FLASHINFER_MOE_FP4"] = "0"
        os.environ["VLLM_USE_FLASHINFER_MOE_INT4"] = "0"

    protobuf_status = enforce_protobuf_runtime_compat()
    print("Protobuf compatibility status:", json.dumps(protobuf_status, indent=2))

    from vllm import LLM, SamplingParams
    import inspect

    candidate_kwargs: Dict[str, Any] = {
        "model": str(model_path),
        "tokenizer": str(model_path),
        "dtype": profile["dtype"],
        "max_model_len": profile["max_model_len"],
        "gpu_memory_utilization": profile["gpu_memory_utilization"],
        "tensor_parallel_size": profile["tensor_parallel_size"],
        "max_num_seqs": profile["max_num_seqs"],
        "max_num_batched_tokens": profile["max_num_batched_tokens"],
        "cpu_offload_gb": profile["cpu_offload_gb"],
        "enforce_eager": profile["enforce_eager"],
        "trust_remote_code": profile["trust_remote_code"],
        "download_dir": str(HF_CACHE_DIR),
        "quantization": profile["quantization"],
    }

    init_signature = inspect.signature(LLM.__init__)
    accepted_params = set(init_signature.parameters.keys())

    if "attention_backend" in accepted_params:
        candidate_kwargs["attention_backend"] = profile["attention_backend"]
    elif "attention_config" in accepted_params:
        try:
            from vllm.config import AttentionConfig

            candidate_kwargs["attention_config"] = AttentionConfig(
                backend=profile["attention_backend"]
            )
        except Exception as exc:
            print(
                f"Attention config setup warning: {exc}. Falling back to automatic backend selection."
            )
    llm_kwargs = {
        key: value
        for key, value in candidate_kwargs.items()
        if key in accepted_params and value is not None
    }

    print("vLLM init kwargs:")
    print(json.dumps(llm_kwargs, indent=2, default=str))
    log_memory("before_engine_init")

    started = time.time()
    try:
        engine = LLM(**llm_kwargs)
        RUNTIME["engine"] = engine
        RUNTIME["engine_ready"] = True
        RUNTIME["last_init_seconds"] = round(time.time() - started, 2)
        print(f"Engine created in {RUNTIME['last_init_seconds']}s")
        log_memory("after_engine_init")

        if run_warmup:
            warmup_params = SamplingParams(
                max_tokens=32,
                temperature=0.0,
                top_p=1.0,
                repetition_penalty=1.0,
            )
            warmup_output = engine.generate(["Reply with exactly: READY"], warmup_params, use_tqdm=False)
            preview = warmup_output[0].outputs[0].text.strip() if warmup_output and warmup_output[0].outputs else ""
            print(f"Warmup output: {preview}")
            log_memory("after_warmup")

        write_runtime_status(
            {
                "phase": "engine_initialized",
                "model_id": MODEL_ID,
                "model_path": str(model_path),
                "profile": profile,
                "init_seconds": RUNTIME["last_init_seconds"],
            }
        )
        return engine

    except RuntimeError as exc:
        error_text = str(exc).lower()
        print("RuntimeError during vLLM init:", exc)
        if "out of memory" in error_text or "cuda" in error_text:
            print(
                "CUDA memory pressure detected. Try lower VLLM_MAX_MODEL_LEN, "
                "VLLM_MAX_NUM_SEQS, or VLLM_GPU_MEMORY_UTILIZATION.",
            )
        if "flashinfer" in error_text or "-lcuda" in error_text:
            print(
                "FlashInfer linker path was detected. Keep VLLM_DISABLE_FLASHINFER=1 for Kaggle runs."
            )
        if "getprototype" in error_text:
            print(
                "protobuf MessageFactory compatibility issue detected. Compat pins + runtime shim were applied."
            )
        print("Profile at failure:", json.dumps(profile, indent=2))
        log_memory("engine_init_failure")
        cleanup_engine(reset_cuda=True, stop_legacy_processes=False)
        raise
    except Exception as exc:
        print("Unexpected exception during vLLM init:", exc)
        traceback.print_exc()
        print("Profile at failure:", json.dumps(profile, indent=2))
        log_memory("engine_init_exception")
        cleanup_engine(reset_cuda=True, stop_legacy_processes=False)
        raise


def generate(
    prompt: str,
    max_tokens: int = 256,
    temperature: float = 0.2,
    top_p: float = 0.9,
    repetition_penalty: float = 1.05,
) -> str:
    if not isinstance(prompt, str) or not prompt.strip():
        raise ValueError("prompt must be a non-empty string")

    if not RUNTIME.get("engine_ready") or RUNTIME.get("engine") is None:
        create_vllm_engine(force_reinit=False, run_warmup=True)

    from vllm import SamplingParams

    safe_max_tokens = max(1, min(int(max_tokens), MAX_GENERATE_TOKENS))
    safe_temperature = max(0.0, min(float(temperature), 1.5))
    safe_top_p = max(0.1, min(float(top_p), 1.0))
    safe_repetition_penalty = max(1.0, float(repetition_penalty))

    sampling_params = SamplingParams(
        max_tokens=safe_max_tokens,
        temperature=safe_temperature,
        top_p=safe_top_p,
        repetition_penalty=safe_repetition_penalty,
    )

    log_memory("before_generate")
    try:
        outputs = RUNTIME["engine"].generate([prompt], sampling_params, use_tqdm=False)
        text = ""
        if outputs and outputs[0].outputs:
            text = outputs[0].outputs[0].text.strip()
        log_memory("after_generate")
        return text
    except RuntimeError as exc:
        print("RuntimeError during generation:", exc)
        if "out of memory" in str(exc).lower():
            print(
                "Generation hit CUDA OOM. Try smaller max_tokens, lower max_num_seqs, "
                "or lower max_model_len.",
            )
        log_memory("generate_failure")
        raise
    except Exception as exc:
        print("Unexpected generation failure:", exc)
        traceback.print_exc()
        log_memory("generate_exception")
        raise

In [ ]:
install_dependencies(force=False)
print("Dependency install phase complete.")

In [ ]:
stage("Model Lock + Auth + Prepare Artifacts")
assert_model_lock()
print(json.dumps(configure_hf_auth(), indent=2))

force_model_refresh = os.getenv("FORCE_MODEL_REFRESH", "0").strip().lower() in {"1", "true", "yes"}
model_path = prepare_model_artifacts(force_download=force_model_refresh)
print(f"MODEL_PATH: {model_path}")
log_memory("after_model_prepare")

In [ ]:
stage("Initialize Local vLLM Engine")
force_reinit = os.getenv("FORCE_ENGINE_REINIT", "0").strip().lower() in {"1", "true", "yes"}
engine = create_vllm_engine(force_reinit=force_reinit, run_warmup=True)

print("Engine ready:", RUNTIME.get("engine_ready"))
print("Init seconds:", RUNTIME.get("last_init_seconds"))

In [ ]:
stage("Interactive Test Calls")
if not RUNTIME.get("engine_ready") or RUNTIME.get("engine") is None:
    create_vllm_engine(force_reinit=False, run_warmup=True)

example_prompts = [
    "Solve briefly: If x + 7 = 19, what is x?",
    "Write one short tip for avoiding algebra sign mistakes.",
]

for idx, prompt in enumerate(example_prompts, start=1):
    print(f"\n--- Example {idx} prompt ---")
    print(prompt)
    answer = generate(prompt, max_tokens=96, temperature=0.2, top_p=0.9)
    print("--- Example output ---")
    print(answer[:1200])

In [ ]:
stage("Runtime Diagnostics")
runtime_snapshot = {
    "dependencies_ready": RUNTIME.get("dependencies_ready"),
    "model_prepared": RUNTIME.get("model_prepared"),
    "engine_ready": RUNTIME.get("engine_ready"),
    "model_path": str(RUNTIME.get("model_path")) if RUNTIME.get("model_path") else None,
    "last_init_seconds": RUNTIME.get("last_init_seconds"),
    "profile": RUNTIME.get("last_profile", {}),
}
print(json.dumps(runtime_snapshot, indent=2))
log_memory("runtime_diagnostics")

print("Manual call example:")
print("generate('Explain prime numbers in 2 sentences.', max_tokens=120)")

## Cleanup and Reset
Run the next cell before re-initializing vLLM or when memory pressure grows.

It will:
- stop optional legacy background processes if present
- release the in-notebook vLLM engine reference
- run garbage collection and clear CUDA cache

In [ ]:
stage("Cleanup and Reset")
cleanup_engine(reset_cuda=True, stop_legacy_processes=True)
write_runtime_status({"phase": "cleanup", "engine_ready": False})
print("Reset complete. Re-run the initialization cell before the next generation call.")